# Format GSE127465 human single-cell data
Read every HUMAN raw-count TSV, restrict the gene panel to its intersection with `st_concat.h5ad`, attach study metadata by library and barcode, and combine the libraries into one sparse AnnData object. All libraries and cells are retained.

In [1]:
from pathlib import Path
import re

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

single_cell_dir = Path('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/single-cell')
metadata_file = single_cell_dir / 'GSE127465_human_cell_metadata_54773x25.tsv'
spatial_file = single_cell_dir.parent / 'spatial' / 'st_concat.h5ad'
output_file = single_cell_dir / 'sc_concat.h5ad'
chunk_size = 100

count_files = sorted(single_cell_dir.glob('GSM*_human_*_raw_counts.tsv'))
assert count_files, f'No HUMAN raw-count TSVs found in {single_cell_dir}'
[path.name for path in count_files]

['GSM3635278_human_p1t1_raw_counts.tsv',
 'GSM3635279_human_p1t2_raw_counts.tsv',
 'GSM3635280_human_p1t3_raw_counts.tsv',
 'GSM3635281_human_p1t4_raw_counts.tsv',
 'GSM3635282_human_p1b1_raw_counts.tsv',
 'GSM3635283_human_p1b2_raw_counts.tsv',
 'GSM3635284_human_p1b3_raw_counts.tsv',
 'GSM3635285_human_p2t1_raw_counts.tsv',
 'GSM3635286_human_p2t2_raw_counts.tsv',
 'GSM3635287_human_p2b1_raw_counts.tsv',
 'GSM3635288_human_p3t1_raw_counts.tsv',
 'GSM3635289_human_p3t2_raw_counts.tsv',
 'GSM3635290_human_p3t3_raw_counts.tsv',
 'GSM3635291_human_p3b1_raw_counts.tsv',
 'GSM3635292_human_p4t1_raw_counts.tsv',
 'GSM3635293_human_p4t2_raw_counts.tsv',
 'GSM3635294_human_p4t3_raw_counts.tsv',
 'GSM3635295_human_p4b1_raw_counts.tsv',
 'GSM3635296_human_p5t1_raw_counts.tsv',
 'GSM3635297_human_p5t2_raw_counts.tsv',
 'GSM3635298_human_p6t1_raw_counts.tsv',
 'GSM3635299_human_p6t2_raw_counts.tsv',
 'GSM3635300_human_p6b1_raw_counts.tsv',
 'GSM3635301_human_p7t1_raw_counts.tsv',
 'GSM3635302_hum

In [2]:
cell_metadata = pd.read_csv(metadata_file, sep='\t', low_memory=False)
metadata_keys = ['Library', 'Barcode']
assert cell_metadata[metadata_keys].notna().all().all()
assert not cell_metadata.duplicated(metadata_keys).any(), f'Duplicate keys in {metadata_file}'

library_pattern = re.compile(r'^(GSM\d+)_human_(.+)_raw_counts\.tsv$')
file_info = []
for path in count_files:
    match = library_pattern.match(path.name)
    if match is None:
        raise ValueError(f'Unexpected HUMAN count filename: {path.name}')
    file_info.append({'path': path, 'GSM_ID': match.group(1), 'Library': match.group(2)})

file_libraries = {item['Library'] for item in file_info}
metadata_libraries = set(cell_metadata['Library'].unique())
assert file_libraries == metadata_libraries, {
    'missing_count_files': sorted(metadata_libraries - file_libraries),
    'files_without_metadata': sorted(file_libraries - metadata_libraries),
}

# Restrict only the gene panel. All libraries and cell rows are retained.
spatial_backed = ad.read_h5ad(spatial_file, backed='r')
spatial_genes = spatial_backed.var_names.tolist()
spatial_backed.file.close()
single_cell_genes = pd.read_csv(count_files[0], sep='\t', nrows=0).columns[1:].tolist()
single_cell_gene_set = set(single_cell_genes)
shared_genes = [gene for gene in spatial_genes if gene in single_cell_gene_set]
assert shared_genes, 'The spatial and single-cell datasets have no shared genes'
for item in file_info:
    header = pd.read_csv(item['path'], sep='\t', nrows=0).columns.tolist()
    if header[0] != 'barcode' or header[1:] != single_cell_genes:
        raise ValueError(f"Gene panel or order differs in {item['path']}")
print(f'{len(shared_genes):,} shared genes ({len(spatial_genes):,} spatial; {len(single_cell_genes):,} single-cell)')
cell_metadata.groupby('Library', observed=True).size()

958 shared genes (980 spatial; 41,861 single-cell)


Library
p1b1    1961
p1b2    2537
p1b3    2628
p1t1    2528
p1t2    2832
p1t3    1344
p1t4    1586
p2b1    4674
p2t1     257
p2t2     235
p3b1    1066
p3t1    3257
p3t2    3383
p3t3    3639
p4b1     447
p4t1    1572
p4t2    1498
p4t3    1886
p5t1    2015
p5t2    2561
p6b1     338
p6t1    2985
p6t2    2687
p7b1     760
p7t1    3210
p7t2    2887
dtype: int64

In [3]:
def load_count_library(path, library, gsm_id, cell_metadata, shared_genes, chunk_size=100):
    matrices, barcodes = [], []
    selected_columns = ['barcode', *shared_genes]
    for chunk in pd.read_csv(path, sep='\t', usecols=selected_columns, chunksize=chunk_size):
        if chunk.columns[0] != 'barcode':
            raise ValueError(f'First column in {path} is not barcode')
        barcode = chunk.iloc[:, 0].astype(str)
        values = chunk[shared_genes].to_numpy(dtype=np.int32, copy=False)
        if (values < 0).any():
            raise ValueError(f'Negative counts found in {path}')
        barcodes.extend(barcode.tolist())
        matrices.append(sparse.csr_matrix(values))

    X = sparse.vstack(matrices, format='csr')
    obs = pd.DataFrame({'Library': library, 'Barcode': barcodes})
    if obs['Barcode'].duplicated().any():
        raise ValueError(f'Duplicate barcodes within {path}')
    obs['_expr_order'] = np.arange(len(obs))
    library_metadata = cell_metadata.loc[cell_metadata['Library'].eq(library)]
    obs = obs.merge(library_metadata, on=['Library', 'Barcode'], how='left', validate='one_to_one', sort=False)
    obs = obs.sort_values('_expr_order').drop(columns='_expr_order').reset_index(drop=True)
    obs.insert(0, 'GSM_ID', gsm_id)
    obs.insert(1, 'source_count_file', str(path))
    obs['_has_cell_metadata'] = obs['Patient'].notna()
    obs['shared_panel_total_counts'] = np.asarray(X.sum(axis=1)).ravel()
    obs.index = pd.Index(library + ':' + obs['Barcode'], name='cell_key')
    assert obs.index.is_unique and len(obs) == X.shape[0]
    for column in obs.select_dtypes(include='object'):
        obs[column] = obs[column].fillna('').astype(str)

    var = pd.DataFrame(index=pd.Index(shared_genes, name='gene_symbol'))
    adata = ad.AnnData(X=X, obs=obs, var=var)
    adata.uns['source_count_file'] = str(path)
    adata.uns['source_metadata_file'] = str(metadata_file)
    return adata

In [4]:
library_adatas = []
for item in file_info:
    print(f"Loading {item['Library']} from {item['path'].name} ...", flush=True)
    library_adata = load_count_library(
        item['path'], item['Library'], item['GSM_ID'], cell_metadata, shared_genes,
        chunk_size=chunk_size,
    )
    print(f"  {library_adata.n_obs:,} cells x {library_adata.n_vars:,} genes; {library_adata.obs['_has_cell_metadata'].sum():,} metadata matches")
    library_adatas.append(library_adata)

# AnnData supports join='inner' or 'outer'. Check the ordered panel
# explicitly before concatenation so an inner join cannot hide a mismatch.
expected_var_names = library_adatas[0].var_names
assert all(a.var_names.equals(expected_var_names) for a in library_adatas), 'Ordered gene panels differ between libraries'
sc_data = ad.concat(library_adatas, axis=0, join='inner', merge='same', uns_merge='unique', index_unique=None)
assert sc_data.var_names.equals(expected_var_names)
assert sc_data.obs_names.is_unique
sc_data.uns['source_metadata_file'] = str(metadata_file)
sc_data.uns['source_spatial_gene_panel_file'] = str(spatial_file)
sc_data.uns['source_count_files'] = [str(item['path']) for item in file_info]
sc_data

Loading p1t1 from GSM3635278_human_p1t1_raw_counts.tsv ...
  5,755 cells x 958 genes; 2,528 metadata matches
Loading p1t2 from GSM3635279_human_p1t2_raw_counts.tsv ...
  6,377 cells x 958 genes; 2,832 metadata matches
Loading p1t3 from GSM3635280_human_p1t3_raw_counts.tsv ...
  4,266 cells x 958 genes; 1,344 metadata matches
Loading p1t4 from GSM3635281_human_p1t4_raw_counts.tsv ...
  4,845 cells x 958 genes; 1,586 metadata matches
Loading p1b1 from GSM3635282_human_p1b1_raw_counts.tsv ...
  3,519 cells x 958 genes; 1,961 metadata matches
Loading p1b2 from GSM3635283_human_p1b2_raw_counts.tsv ...
  4,270 cells x 958 genes; 2,537 metadata matches
Loading p1b3 from GSM3635284_human_p1b3_raw_counts.tsv ...
  4,974 cells x 958 genes; 2,628 metadata matches
Loading p2t1 from GSM3635285_human_p2t1_raw_counts.tsv ...
  7,195 cells x 958 genes; 257 metadata matches
Loading p2t2 from GSM3635286_human_p2t2_raw_counts.tsv ...
  7,678 cells x 958 genes; 235 metadata matches
Loading p2b1 from GSM36

AnnData object with n_obs × n_vars = 173954 × 958
    obs: 'GSM_ID', 'source_count_file', 'Library', 'Barcode', 'Patient', 'Tissue', 'Barcoding emulsion', 'Total counts', 'Percent counts from mitochondrial genes', 'Most likely LM22 cell type', 'Major cell type', 'Minor subset', 'used_in_NSCLC_all_cells', 'x_NSCLC_all_cells', 'y_NSCLC_all_cells', 'used_in_NSCLC_and_blood_immune', 'x_NSCLC_and_blood_immune', 'y_NSCLC_and_blood_immune', 'used_in_NSCLC_immune', 'x_NSCLC_immune', 'y_NSCLC_immune', 'used_in_NSCLC_non_immune', 'x_NSCLC_non_immune', 'y_NSCLC_non_immune', 'used_in_blood', 'x_blood', 'y_blood', '_has_cell_metadata', 'shared_panel_total_counts'
    uns: 'source_metadata_file', 'source_spatial_gene_panel_file', 'source_count_files'

In [7]:
sc_data.n_obs

173954

In [8]:
summary = sc_data.obs.groupby('Library', observed=True).agg(
    cells=('Barcode', 'size'),
    cells_with_metadata=('_has_cell_metadata', 'sum'),
    shared_panel_total_counts=('shared_panel_total_counts', 'sum'),
    metadata_total_counts=('Total counts', 'sum'),
)
display(summary)
assert sc_data.shape == (len(sc_data.obs), len(sc_data.var))
# assert sc_data.n_obs == len(cell_metadata) == 54_773
# assert sc_data.obs['_has_cell_metadata'].all()
assert np.isfinite(sc_data.X.data).all() and (sc_data.X.data >= 0).all()
sc_data.write_h5ad(output_file, compression='gzip')
print(f'Wrote {sc_data.n_obs:,} cells x {sc_data.n_vars:,} genes to {output_file}')

,cells,cells_with_metadata,shared_panel_total_counts,metadata_total_counts
Library,,,,
p1b1,3519,1961,589921,4085245.0
p1b2,4270,2537,707700,4770827.0
p1b3,4974,2628,732911,4823979.0
p1t1,5755,2528,1215048,6258500.0
p1t2,6377,2832,1311616,6720313.0
p1t3,4266,1344,898513,4972610.0
p1t4,4845,1586,907889,5029808.0
p2b1,15826,4674,827020,4931660.0
p2t1,7195,257,154265,773640.0


Wrote 173,954 cells x 958 genes to /work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/single-cell/sc_concat.h5ad


In [23]:
sc_data[sc_data.obs['Tissue']=='tumor'].write_h5ad(output_file, compression='gzip')

In [27]:
sorted(sc_data[sc_data.obs['Tissue']=='tumor'].obs['Major cell type'].unique())

['Ciliated cells',
 'Club cells',
 'Endothelial cells',
 'Fibroblasts',
 'ND',
 'Patient1-specific',
 'Patient2-specific',
 'Patient3-specific',
 'Patient4-specific',
 'Patient5-specific',
 'Patient6-specific',
 'Patient7-specific',
 'Smooth muscle cells',
 'Type I cells',
 'Type II cells',
 'tB cells',
 'tMast cells',
 'tMoMacDC',
 'tNK cells',
 'tNeutrophils',
 'tPlasma cells',
 'tRBC',
 'tT cells',
 'tpDC']